In [1]:
!pip install tensorflow keras split-folders scikit-learn scikit-image seaborn opencv-python-headless #opencv-python

# !mkdir /food
# !cd food
# !wget http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
# !tar xzvf food-101.tar.gz
# !cd
# !ls

# !!pip uninstall -y opencv-python opencv-contrib-python
# !pip install opencv-python-headless

# import subprocess
# import os
# subprocess.run(['ln', '-sf', '/usr/lib/x86_64-linux-gnu/libcudnn.so.8',
#                 '/usr/lib/x86_64-linux-gnu/libcudnn.so.9'], check=True)
# subprocess.run(['ldconfig'], check=True)
# # Check what's actually missing
# result = subprocess.run(['ldconfig', '-p'], capture_output=True, text=True)
# cudnn_libs = [l for l in result.stdout.split('\n') if 'cudnn' in l]
# print('\n'.join(cudnn_libs))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 23.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 23.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 25.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 25.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 25.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import os
import shutil
import stat
import seaborn as sns
import collections
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from os import listdir
from os.path import join
from collections import defaultdict
import keras
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, ReduceLROnPlateau

#Limit GPU VRAM allocation (to prevent memory fatigs 
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

!nvidia-smi
!nvcc --version
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))
!python -c "import tensorflow as tf; print(tf.__version__)"

2026-05-01 18:25:00.996150: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777659901.113045      24 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777659901.151788      24 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-01 18:25:01.454251: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Fri May  1 18:25:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.52.01              Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 Ti     On  |   00000000:06:00.0  On |                  N/A |
|  0%   44C    P8             17W /  220W |     582MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
import shutil
import stat
import seaborn as sns
import collections
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from os import listdir
from os.path import join
from collections import defaultdict
import keras

#InceptionV3 model
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
#Updating model from inceptionV3 -> EfficentNetV2S as it has more modern architecture

# from keras.applications import EfficientNetV2S
# from keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, ReduceLROnPlateau

#Model Main Logic

MODEL_NAME = "InceptionV3_v0.0"
BASE_DIR    = "/tf/data/models"
SAVE_DIR    = f"{BASE_DIR}/{MODEL_NAME}"
os.makedirs(SAVE_DIR, exist_ok=True)

EPOCHS = 30
FINE_TUNE_EPOCHS = 20
IMG_SIZE = (299, 299)
BATCH_SIZE = 16
N_CLASSES = 101
AUTOTUNE = tf.data.AUTOTUNE

#Location to save model after each epoch
ENV = None
# ENV = "cloud"
ENV = "local"

if ENV == "cloud":
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir("/content"):
        raise FileNotFoundError("/content does not exist")
    if not os.path.isdir("/content/food-101"):
        %cd /content
        !wget http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
        !tar xzvf food-101.tar.gz
    DATA_ROOT = "/content/food-101"
    # SAVE_DIR = '/content/drive/MyDrive/ML/final/models'

elif ENV == "local":
    if not os.path.isdir("/tf/data"):
        raise FileNotFoundError("/tf/data does not exist")
    DATA_ROOT = "/tf/data/food-101"
    # SAVE_DIR = '/tf/data/models/'
    os.makedirs(SAVE_DIR, exist_ok=True)

# elif os.path.isdir("/content"):
#     if not os.path.isdir("/content/food-101"):
#         %cd /content
#         !wget http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
#         !tar xzvf food-101.tar.gz
#     DATA_ROOT = "/content/food-101"

# elif os.path.isdir("/tf/data"):
#     DATA_ROOT = "/tf/data/food-101"

else:
    raise FileNotFoundError("Could not find /content or /tf/data")

TRAIN_DIR = f"{DATA_ROOT}/train"
TEST_DIR = f"{DATA_ROOT}/test"

class_N = {}
N_class = {}
with open(f'{DATA_ROOT}/meta/classes.txt', 'r') as txt:
    classes = [i.strip() for i in txt.readlines()]
    class_N = dict(zip(classes, range(len(classes))))
    N_class = dict(zip(range(len(classes)), classes))
    class_N = {i: j for j, i in N_class.items()}
print(class_N)

# Method to generate directory-file map.
def gen_dir_file_map(path):
    dir_files = defaultdict(list)
    with open(path, 'r') as txt:
        files = [i.strip() for i in txt.readlines()]
        for f in files:
            dir_name, id = f.split('/')
            dir_files[dir_name].append(id + '.jpg')
    return dir_files

# Method to recursively copy a directory.
def copytree(source, target, symlinks = False, ignore = None):
  if not os.path.exists(target):
      os.makedirs(target)
      shutil.copystat(source, target)
  data = os.listdir(source)
  if ignore:
      exclude = ignore(source, data)
      data = [x for x in data if x not in exclude]
  for item in data:
      src = os.path.join(source, item)
      dest = os.path.join(target, item)
      if symlinks and os.path.islink(src):
          if os.path.lexists(dest):
              os.remove(dest)
          os.symlink(os.readlink(src), dest)
          try:
              st = os.lstat(src)
              mode = stat.S_IMODE(st.st_mode)
              os.lchmod(dest, mode)
          except:
              pass
      elif os.path.isdir(src):
          copytree(src, dest, symlinks, ignore)
      else:
          shutil.copy2(src, dest)

# Train files to ignore.
def ignore_train(d, filenames):
  subdir = d.split('/')[-1]
  train_dir_files = gen_dir_file_map(f'{DATA_ROOT}/meta/train.txt')
  to_ignore = train_dir_files[subdir]
  return to_ignore

# Test files to ignore.
def ignore_test(d, filenames):
  subdir = d.split('/')[-1]
  test_dir_files = gen_dir_file_map(f'{DATA_ROOT}/meta/test.txt')
  to_ignore = test_dir_files[subdir]
  return to_ignore

# Method to generate train-test files.
def gen_train_test_split(path_to_imgs = f'{DATA_ROOT}/images' , target_path = DATA_ROOT):
  copytree(path_to_imgs, target_path + '/train', ignore=ignore_test)
  copytree(path_to_imgs, target_path + '/test', ignore=ignore_train)

# Generate train-test files.
if not os.path.isdir(f'{DATA_ROOT}/test') and not os.path.isdir(f'{DATA_ROOT}/train'):
    gen_train_test_split()
    len_train = len(os.listdir(f'{DATA_ROOT}/train'))
    len_test = len(os.listdir(f'{DATA_ROOT}/test')) # Fixed path from food-101 to food-101
    print('Train/Test dirs generated: len_train ='+str(len_train)+'len_test='+str(len_test))
else:
    print('train and test folders already exists.')
    len_train = len(os.listdir(f'{DATA_ROOT}/train'))
    len_test = len(os.listdir(f'{DATA_ROOT}/test'))
    print(len_train,len_test)

# Write model_info.txt
def save_model_info(model, save_dir):
    import datetime

    optimizer  = model.optimizer
    loss       = model.loss
    base_model = next((l for l in model.layers if isinstance(l, tf.keras.Model)), None)

    # Optimizer info 
    opt_name = optimizer.__class__.__name__
    try:
        lr = optimizer.learning_rate
        lr_str = f"{float(lr):.2e}" if hasattr(lr, '__float__') else lr.__class__.__name__
    except:
        lr_str = "unknown"

    # Loss info 
    if hasattr(loss, '__class__'):
        loss_name = loss.__class__.__name__
        loss_cfg  = loss.get_config() if hasattr(loss, 'get_config') else {}
    else:
        loss_name = str(loss)
        loss_cfg  = {}

    # Base model info 
    base_name      = base_model.name if base_model else "none"
    total_layers   = len(base_model.layers) if base_model else 0
    frozen_layers  = sum(1 for l in (base_model.layers if base_model else []) if not l.trainable)
    trainable_layers = total_layers - frozen_layers

    #  Head layers (everything after the base model) 
    head_layers = [l for l in model.layers if l is not base_model and not isinstance(l, tf.keras.layers.InputLayer)]
    head_str    = " → ".join(
        f"{l.__class__.__name__}({_layer_args(l)})"
        for l in head_layers
    )

    # Augmentation - scan augment() source 
    import inspect
    try:
        aug_src   = inspect.getsource(augment)
        aug_ops   = [line.strip() for line in aug_src.splitlines()
                     if 'tf.image.' in line and not line.strip().startswith('#')]
    except:
        aug_ops = ["(could not extract)"]

    # -- Write file ----------------------------------------------------------
    info_path = f"{save_dir}/{MODEL_NAME}_model_info.txt"
    with open(info_path, "w") as f:
        f.write(f"DATE              : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"MODEL_NAME        : {MODEL_NAME}\n")
        f.write(f"EPOCHS            : {EPOCHS}\n")
        f.write(f"FINE_TUNE_EPOCHS  : {FINE_TUNE_EPOCHS}\n")
        f.write(f"IMG_SIZE          : {IMG_SIZE}\n")
        f.write(f"BATCH_SIZE        : {BATCH_SIZE}\n")
        f.write(f"N_CLASSES         : {N_CLASSES}\n")
        f.write(f"\n-- Optimizer------------------------------------------\n")
        f.write(f"  class           : {opt_name}\n")
        f.write(f"  learning_rate   : {lr_str}\n")
        f.write(f"\n-- Loss-----------------------------------------------\n")
        f.write(f"  class           : {loss_name}\n")
        for k, v in loss_cfg.items():
            f.write(f"  {k:<16}  : {v}\n")
        f.write(f"\n-- Base Model ----------------------------------------\n")
        f.write(f"  name            : {base_name}\n")
        f.write(f"  total layers    : {total_layers}\n")
        f.write(f"  frozen layers   : {frozen_layers}\n")
        f.write(f"  trainable layers: {trainable_layers}\n")
        f.write(f"\n-- Head ----------------------------------------------\n")
        f.write(f"  {head_str}\n")
        f.write(f"\n-- Augmentation ops ----------------------------------\n")
        for op in aug_ops:
            f.write(f"  {op}\n")
        f.write(f"\n-- Full Layer Summary --------------------------------\n")
        model.summary(print_fn=lambda line: f.write(line + "\n"), show_trainable=True)

    print(f"model_info.txt saved → {info_path}")


def _layer_args(layer):
    """Extract the most useful arg from a layer for the head summary string."""
    cfg = layer.get_config()
    if 'units' in cfg:
        s = str(cfg['units'])
        if cfg.get('activation'): s += f", {cfg['activation']}"
        if cfg.get('kernel_regularizer'): s += ", l2"
        return s
    if 'rate' in cfg:
        return str(cfg['rate'])
    if 'axis' in cfg:
        return f"axis={cfg['axis']}"
    return ""

@tf.function
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
  # image = tf.image.random_flip_up_down(image)     #might not be as usefull as the other
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.05)   # slight hue jitter

    # Pad 10% then random crop back to IMG_SIZE (replaces RandomZoom/Translate)
    pad_h = int(IMG_SIZE[0] * 0.1)
    pad_w = int(IMG_SIZE[1] * 0.1)
    image = tf.image.pad_to_bounding_box(
        image, pad_h, pad_w,
        IMG_SIZE[0] + 2 * pad_h,
        IMG_SIZE[1] + 2 * pad_w
    )
    # Fix: Include batch dimension in random_crop size
    image = tf.image.random_crop(image, size=[tf.shape(image)[0], IMG_SIZE[0], IMG_SIZE[1], 3])
    return image, label



def build_dataset(directory, augment_data=False, prefetch_size=2):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, label_mode='categorical'
    )
    if augment_data:
        # ds = ds.shuffle(buffer_size=500, reshuffle_each_iteration=True)  # shuffle data
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE) # change AUTOTUNE IF MEMOMORY NOT ENOUGH
    ds = ds.map(lambda x, y: (preprocess_input(x), y),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(prefetch_size)

train_ds = build_dataset(TRAIN_DIR, augment_data=True)
test_ds  = build_dataset(TEST_DIR,  augment_data=False)

{'apple_pie': 0, 'baby_back_ribs': 1, 'baklava': 2, 'beef_carpaccio': 3, 'beef_tartare': 4, 'beet_salad': 5, 'beignets': 6, 'bibimbap': 7, 'bread_pudding': 8, 'breakfast_burrito': 9, 'bruschetta': 10, 'caesar_salad': 11, 'cannoli': 12, 'caprese_salad': 13, 'carrot_cake': 14, 'ceviche': 15, 'cheesecake': 16, 'cheese_plate': 17, 'chicken_curry': 18, 'chicken_quesadilla': 19, 'chicken_wings': 20, 'chocolate_cake': 21, 'chocolate_mousse': 22, 'churros': 23, 'clam_chowder': 24, 'club_sandwich': 25, 'crab_cakes': 26, 'creme_brulee': 27, 'croque_madame': 28, 'cup_cakes': 29, 'deviled_eggs': 30, 'donuts': 31, 'dumplings': 32, 'edamame': 33, 'eggs_benedict': 34, 'escargots': 35, 'falafel': 36, 'filet_mignon': 37, 'fish_and_chips': 38, 'foie_gras': 39, 'french_fries': 40, 'french_onion_soup': 41, 'french_toast': 42, 'fried_calamari': 43, 'fried_rice': 44, 'frozen_yogurt': 45, 'garlic_bread': 46, 'gnocchi': 47, 'greek_salad': 48, 'grilled_cheese_sandwich': 49, 'grilled_salmon': 50, 'guacamole': 5

I0000 00:00:1777659911.524100      24 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:06:00.0, compute capability: 8.6


Found 25250 files belonging to 101 classes.


In [4]:
# VALIDATING for no corrupt data
# import os
# import PIL.Image
# from pathlib import Path
# import cv2
# from tqdm import tqdm

# print("Starting unified deep scan for corrupted images...")

# all_images = list(Path(DATA_ROOT).rglob('*.jpg'))
# removed_count = 0

# for img_path in tqdm(all_images):
#     corrupted = False
#     # 1. Quick Header Check
#     try:
#         with PIL.Image.open(img_path) as img:
#             img.verify()
#     except (IOError, SyntaxError):
#         corrupted = True

#     # 2. Deep Decode Check (if header passed)
#     if not corrupted:
#         img_cv = cv2.imread(str(img_path))
#         if img_cv is None:
#             corrupted = True

#     if corrupted:
#         print(f'\nRemoving corrupted image: {img_path}')
#         try:
#             os.remove(img_path)
#             removed_count += 1
#         except Exception as e:
#             print(f"Failed to remove {img_path}: {e}")

# print(f"\nDeep scan finished. Total removed: {removed_count} images.")

In [5]:
# TRAINING
base_model = InceptionV3(weights='imagenet', include_top=False,
                         input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
# training=False: keeps InceptionV3's BatchNorm layers in inference mode
# while frozen — prevents corrupting pretrained running statistics
# Dense Layers
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
1
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(N_CLASSES,
                       kernel_regularizer=regularizers.l2(0.005),
                       activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary(show_trainable=True)
save_model_info(model, SAVE_DIR) 

print("\n=== Phase 1: Head training ===")
history_ht = model.fit(
    train_ds, validation_data=test_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(f'{SAVE_DIR}/{MODEL_NAME}_best_base.keras',
                        save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6),  # ← NEW
        CSVLogger(f'{SAVE_DIR}/{MODEL_NAME}_history_base.log'),
    ]
)
model.save(f'{SAVE_DIR}/{MODEL_NAME}_final_base.keras')
print("Saved Trained Model.")

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 299, 299, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ inception_v3 (Functional)   │ (None, 8, 8, 2048)    │ 21,802,784 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ global_average_pooling2d    │ (None, 2048)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 256)           │    524,544 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout (Dropout)           │ (None, 256)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_1 (Dense)             │ (None, 101)           │     25,957 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 22,353,285 (85.27 MB)

 Trainable params: 550,501 (2.10 MB)

...

4733/4735 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2647 - loss: 3.3242

2026-05-01 18:28:32.636167: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5617_0', 8 bytes spill stores, 8 bytes spill loads

2026-05-01 18:28:32.966771: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8951', 620 bytes spill stores, 620 bytes spill loads

2026-05-01 18:28:33.017960: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8951', 556 bytes spill stores, 556 bytes spill loads

2026-05-01 18:28:33.197142: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8949', 104 bytes spill stores, 104 bytes spill loads



4735/4735 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2647 - loss: 3.3240

2026-05-01 18:29:23.845526: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2453_0', 8 bytes spill stores, 8 bytes spill loads



4735/4735 ━━━━━━━━━━━━━━━━━━━━ 250s 49ms/step - accuracy: 0.2648 - loss: 3.3239 - val_accuracy: 0.5230 - val_loss: 1.9765 - learning_rate: 0.0010
Epoch 2/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 216s 46ms/step - accuracy: 0.4198 - loss: 2.4124 - val_accuracy: 0.5499 - val_loss: 1.8491 - learning_rate: 0.0010
Epoch 3/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 221s 47ms/step - accuracy: 0.4492 - loss: 2.2890 - val_accuracy: 0.5688 - val_loss: 1.7690 - learning_rate: 0.0010
Epoch 4/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 220s 46ms/step - accuracy: 0.4620 - loss: 2.2273 - val_accuracy: 0.5786 - val_loss: 1.7050 - learning_rate: 0.0010
Epoch 5/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 208s 44ms/step - accuracy: 0.4755 - loss: 2.1727 - val_accuracy: 0.5855 - val_loss: 1.6796 - learning_rate: 0.0010
Epoch 6/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 211s 45ms/step - accuracy: 0.4790 - loss: 2.1430 - val_accuracy: 0.5882 - val_loss: 1.6595 - learning_rate: 0.0010
Epoch 7/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 215s 45ms/step - accuracy: 

In [6]:
# ── Phase 2: Fine-tuning ──────────────────────────────────────────────────────
# Unfreeze ONLY the last 2 Inception blocks (layers 249+), not a raw count of 50

base_model.trainable = True
for layer in base_model.layers[:249]:
    layer.trainable = False
# Keep BN layers in inference mode to protect pretrained stats
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

steps_per_epoch = len(train_ds)
total_steps = FINE_TUNE_EPOCHS * steps_per_epoch

# Cosine decay from 1e-5 → 1e-7 (critical: much lower than Phase 1)
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=total_steps,
    alpha=1e-7
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')]
)

history_ft = model.fit(
    train_ds, validation_data=test_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[
        ModelCheckpoint(f'{SAVE_DIR}/{MODEL_NAME}_best_finetuned.keras',
                        save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(patience=8, restore_best_weights=True),
        CSVLogger(f'{SAVE_DIR}/{MODEL_NAME}_history_finetuned.log'),
    ]
)
model.save(f'{SAVE_DIR}/{MODEL_NAME}_final.keras')

Epoch 1/20
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 298s 58ms/step - accuracy: 0.5756 - loss: 2.4046 - top5_acc: 0.8286 - val_accuracy: 0.6759 - val_loss: 2.0402 - val_top5_acc: 0.8958
Epoch 2/20
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 263s 55ms/step - accuracy: 0.6160 - loss: 2.2583 - top5_acc: 0.8555 - val_accuracy: 0.6940 - val_loss: 1.9771 - val_top5_acc: 0.9027
Epoch 3/20
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 308s 65ms/step - accuracy: 0.6502 - loss: 2.1517 - top5_acc: 0.8737 - val_accuracy: 0.7061 - val_loss: 1.9282 - val_top5_acc: 0.9099
Epoch 4/20
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 272s 57ms/step - accuracy: 0.6783 - loss: 2.0598 - top5_acc: 0.8927 - val_accuracy: 0.7140 - val_loss: 1.8988 - val_top5_acc: 0.9156
Epoch 5/20
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 283s 60ms/step - accuracy: 0.6996 - loss: 1.9914 - top5_acc: 0.9033 - val_accuracy: 0.7185 - val_loss: 1.8799 - val_top5_acc: 0.9169
Epoch 6/20
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 269s 57ms/step - accuracy: 0.7172 - loss: 1.9291 - top5_acc: 0.9147 - val_accuracy:

In [7]:
# ── Terminal Accuracy Report ──────────────────────────────────────────────────
import numpy as np

print("\nRunning inference on test set...")
y_true, y_pred_probs = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_probs.extend(preds)

y_true       = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred       = np.argmax(y_pred_probs, axis=1)

# Top-1
top1 = np.mean(y_true == y_pred) * 100

# Top-5
top5_hits = [y_true[i] in np.argsort(y_pred_probs[i])[-5:] for i in range(len(y_true))]
top5 = np.mean(top5_hits) * 100

# Training history (last epoch)
final_train_acc = history_ht.history['accuracy'][-1] * 100
final_val_acc   = history_ht.history['val_accuracy'][-1] * 100
best_val_acc    = max(history_ht.history['val_accuracy']) * 100

print(f"""
{MODEL_NAME}
Train accuracy (last epoch): {final_train_acc:>6.2f}%  
Val   accuracy (last epoch): {final_val_acc:>6.2f}%  
Best  val accuracy:          {best_val_acc:>6.2f}%  

Test  Top-1 accuracy:        {top1:>6.2f}%  
Test  Top-5 accuracy:        {top5:>6.2f}%  

""")


Running inference on test set...

InceptionV3_v0.0
Train accuracy (last epoch):  57.38%  
Val   accuracy (last epoch):  64.88%  
Best  val accuracy:           64.88%  

Test  Top-1 accuracy:         74.66%  
Test  Top-5 accuracy:         92.45%  




2026-05-01 21:46:26.270344: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Note on the Fix
If the error persists after the deep scan, it might be due to a specific image that is technically readable but has a 'restart marker' issue. We can wrap the dataset in a `filter` to skip images that cause decoding errors, though the deep scan above usually resolves 99% of Food-101 issues.

In [9]:
import os
import tensorflow as tf

primary_path = f"{SAVE_DIR}/{MODEL_NAME}_final.keras"
fallback_path = f"{SAVE_DIR}/{MODEL_NAME}_model_finetuned.keras"

if os.path.exists(primary_path):
    model_path = primary_path
elif os.path.exists(fallback_path):
    model_path = fallback_path
else:
    print("no modules are trained yet")
    raise SystemExit()

model = tf.keras.models.load_model(model_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = f"{SAVE_DIR}/{MODEL_NAME}_food101.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"TFLite model saved to {tflite_path}")
print(f"File size: {os.path.getsize(tflite_path) / 1024 / 1024:.1f} MB")

INFO:tensorflow:Assets written to: /tmp/tmpy8x_ap4s/assets


INFO:tensorflow:Assets written to: /tmp/tmpy8x_ap4s/assets


Saved artifact at '/tmp/tmpy8x_ap4s'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 299, 299, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 101), dtype=tf.float32, name=None)
Captures:
  133062644415376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133062644415952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133057661206800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133064520961488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133057661206608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133057661207376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133057661208528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133057661208336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133057661207568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133057661209104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13305766121

W0000 00:00:1777672883.204203      24 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1777672883.204265      24 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-05-01 22:01:23.204429: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpy8x_ap4s
2026-05-01 22:01:23.222086: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-05-01 22:01:23.222105: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpy8x_ap4s
2026-05-01 22:01:23.365647: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-05-01 22:01:24.149577: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpy8x_ap4s
2026-05-01 22:01:24.324428: I tensorflow/cc/saved_model/loader.cc:466] SavedModel load for tags { serve }; Status: success: OK. Took 1120002 microseconds.


TFLite model saved to /tf/data/models/InceptionV3_v0.0/InceptionV3_v0.0_food101.tflite
File size: 85.2 MB


In [1]:
#Training Curves
import matplotlib.pyplot as plt
import numpy as np

# -- Combine both phases
def combine_histories(h1, h2, key):
    return h1.history[key] + h2.history[key]

epochs_ht = range(1, len(history_ht.history['accuracy']) + 1)
epochs_ft = range(len(epochs_ht) + 1, len(epochs_ht) + len(history_ft.history['accuracy']) + 1)
all_epochs = list(epochs_ht) + list(epochs_ft)
phase_boundary = len(epochs_ht)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} — Training Curves', fontsize=14)

# -- Loss
axes[0].plot(all_epochs, combine_histories(history_ht, history_ft, 'loss'), label='Train')
axes[0].plot(all_epochs, combine_histories(history_ht, history_ft, 'val_loss'), label='Val')
axes[0].axvline(phase_boundary, color='gray', linestyle='--', label='Fine-tune start')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# -- Top-1 Accuracy
axes[1].plot(all_epochs, combine_histories(history_ht, history_ft, 'accuracy'), label='Train')
axes[1].plot(all_epochs, combine_histories(history_ht, history_ft, 'val_accuracy'), label='Val')
axes[1].axvline(phase_boundary, color='gray', linestyle='--', label='Fine-tune start')
axes[1].set_title('Top-1 Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# -- Top-5 Accuracy
axes[2].plot(all_epochs, combine_histories(history_ht, history_ft, 'top5_acc'), label='Train')
axes[2].plot(all_epochs, combine_histories(history_ht, history_ft, 'val_top5_acc'), label='Val')
axes[2].axvline(phase_boundary, color='gray', linestyle='--', label='Fine-tune start')
axes[2].set_title('Top-5 Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/{MODEL_NAME}_training_curves.png', dpi=150)
plt.show()
print(f"Saved → {SAVE_DIR}/{MODEL_NAME}_training_curves.png")

NameError: name 'history_ht' is not defined

In [2]:
# Validation on TEST
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best fine-tuned model (resilient to kernel restart)
try:
    model
except NameError:
    print(f'Loading best_model_finetuned.keras from {SAVE_DIR}/{MODEL_NAME}_final.keras...')
    model = keras.models.load_model(f'{SAVE_DIR}/{MODEL_NAME}_final.keras')
    model.save(f'{SAVE_DIR}/{MODEL_NAME}_final.keras')
# Rebuild test_ds if needed
try:
    test_ds
except NameError:
    test_ds = build_dataset(TEST_DIR, augment_data=False)

# Collect predictions
print("Running inference on test set ...")
y_true, y_pred = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
top1_acc = np.mean(y_true == y_pred)
# Top-5: check if true label is in top-5 predicted indices
y_pred_top5 = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    top5 = np.argsort(preds, axis=1)[:, -5:]
    y_pred_top5.extend(top5.tolist())

true_in_top5 = [y_true[i] in y_pred_top5[i] for i in range(len(y_true))]
top5_acc = np.mean(true_in_top5)

print(f"\n{'='*40}")
print(f"  Top-1 Accuracy : {top1_acc*100:.2f}%")
print(f"  Top-5 Accuracy : {top5_acc*100:.2f}%")
print(f"{'='*40}\n")

# Per-class report (top 20 worst classes)
class_names = [N_class[i] for i in range(N_CLASSES)]
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

per_class_f1 = {cls: report[cls]['f1-score'] for cls in class_names}
worst20 = sorted(per_class_f1.items(), key=lambda x: x[1])[:20]

print("20 worst-performing classes (by F1):")
for cls, f1 in worst20:
    print(f"  {cls:<30} F1={f1:.3f}")

# Confusion matrix (top-20 worst classes only, for readability)
worst_indices = [class_names.index(c) for c, _ in worst20]
mask_true = np.isin(y_true, worst_indices)
cm = confusion_matrix(y_true[mask_true], y_pred[mask_true], labels=worst_indices)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=[class_names[i] for i in worst_indices],
            yticklabels=[class_names[i] for i in worst_indices])
plt.title('Confusion Matrix — 20 Worst Classes')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig(f'{SAVE_DIR}/{MODEL_NAME}_heatmap.png', dpi=150)
plt.show()
print(f"Saved → {SAVE_DIR}/{MODEL_NAME}_heatmap.png")


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Clear CACHE
# import subprocess
# result = subprocess.run(
#     'rm -rf /content/*.tempstate* /content/*.lockfile /content/food-101.tar.gz && df -h /content',
#     shell=True, capture_output=True, text=True
# )
# print(result.stdout)
# print(result.stderr)

In [ ]:
# # CLEAN DEAD FILES
# import subprocess

# # Show the biggest directories eating your disk
# result = subprocess.run(
#     ['du', '-sh', '--threshold=100M',
#      '/content/cache_train',
#      '/content/cache_test',
#      '/content/food-101',
#      '/content/drive/.shortcut-targets-by-id',
#      '/root/.keras',
#      '/tmp'],
#     capture_output=True, text=True
# )
# print(result.stdout)
# print(result.stderr)

# # Also show overall breakdown
# print("\n--- Top space consumers in /content ---")
# subprocess.run(['du', '-sh', '/content/*'], shell=False)
# result2 = subprocess.run('du -sh /content/* 2>/dev/null | sort -rh | head -20',
#                          shell=True, capture_output=True, text=True)
# print(result2.stdout)